In [1]:
import json

import numpy as np
import scipy
from tabulate import tabulate


In [2]:
sleep_metrics_file_anysleep = "table_s4b_anysleep.json"  # generated by table_s4b_anysleep_metrics.ipynb
sleep_metrics_anysleep = json.load(open(sleep_metrics_file_anysleep, "r"))

se_map_anysleep = sleep_metrics_anysleep["se_map"]
se_gt_anysleep = sleep_metrics_anysleep["se_gt"]
waso_map_anysleep = sleep_metrics_anysleep["waso_map"]
waso_gt_anysleep = sleep_metrics_anysleep["waso_gt"]
rsl_map_anysleep = sleep_metrics_anysleep["rsl_map"]
rsl_gt_anysleep = sleep_metrics_anysleep["rsl_gt"]
sol_map_anysleep = sleep_metrics_anysleep["sol_map"]
sol_gt_anysleep = sleep_metrics_anysleep["sol_gt"]

sleep_metrics_file_usleep = "table_s4b_usleep.json"  # generated by table_s4b_usleep_metrics.ipynb
sleep_metrics_usleep = json.load(open(sleep_metrics_file_usleep, "r"))

se_map_usleep = sleep_metrics_usleep["se_map"]
se_gt_usleep = sleep_metrics_usleep["se_gt"]
waso_map_usleep = sleep_metrics_usleep["waso_map"]
waso_gt_usleep = sleep_metrics_usleep["waso_gt"]
rsl_map_usleep = sleep_metrics_usleep["rsl_map"]
rsl_gt_usleep = sleep_metrics_usleep["rsl_gt"]
sol_map_usleep = sleep_metrics_usleep["sol_map"]
sol_gt_usleep = sleep_metrics_usleep["sol_gt"]

In [3]:
def get_avg_std(met_dict, in_hours):
    avg = np.mean(list(met_dict.values()))
    std = np.std(list(met_dict.values()))
    if in_hours:
        avg_neg = avg < 0
        avg_h = int(abs(avg) / 60)
        avg_min = int(np.round(abs(avg) % 60))
        std_h = int(std / 60)
        std_min = int(np.round(std % 60))
        return (f"{'-' if avg_neg else ''}{avg_h}:{str(avg_min).zfill(2)}"
                f"$\\pm${std_h}:{str(std_min).zfill(2)}")
    return f"{avg:.2f}$\\pm${std:.2f}"


def get_spearman(p_met_d, gt_met_d):
    p_agg = {}
    for mr, mr_dict in p_met_d.items():
        for s_id, s_val in mr_dict["1"].items():
            if s_id not in p_agg:
                p_agg[s_id] = []
            p_agg[s_id].append(s_val)

    s_ids = list(sorted(p_agg.keys()))
    p_vals = [np.mean(p_agg[k]) for k in s_ids]
    gt_vals = [gt_met_d[k] for k in s_ids]
    corr, p = scipy.stats.spearmanr(p_vals, gt_vals)
    return f"{corr:.2f}"


table_data = []
for metric_name, p_metric_dict_anysleep, p_metric_dict_usleep, gt_metric_dict in [
    ("Sleep efficiency", se_map_anysleep, se_map_usleep, se_gt_anysleep),
    ("Wake after sleep onset", waso_map_anysleep, waso_map_usleep, waso_gt_anysleep),
    ("REM latency", rsl_map_anysleep, rsl_map_usleep, rsl_gt_anysleep),
    ("Sleep latency", sol_map_anysleep, sol_map_usleep, sol_gt_anysleep),
]:
    print(metric_name)

    # ensure same subjects
    first_mr = list(p_metric_dict_anysleep.keys())[0]
    shared_s_ids = set(p_metric_dict_anysleep[first_mr]["1"].keys()) & set(gt_metric_dict.keys())
    if set(p_metric_dict_anysleep[first_mr]["1"].keys()) != shared_s_ids:
        print(f"Missing keys from pred: {set(p_metric_dict_anysleep[first_mr]['1'].keys()) - shared_s_ids}")
    if set(gt_metric_dict.keys()) != shared_s_ids:
        print(f"Missing keys from ground truth: {set(gt_metric_dict.keys()) - shared_s_ids}")
    # filter to shared subjects
    gt_metric_dict = {k: v for k, v in gt_metric_dict.items() if k in shared_s_ids}
    pred_dict_anysleep = {
        f"{k}_{k2}": v2
        for k, v in p_metric_dict_anysleep.items()
        for k2, v2 in v["1"].items() if k2 in shared_s_ids
    }
    delta_dict_anysleep = {f"{mr}_{s_id}": gt_metric_dict[s_id] - pred_dict_anysleep[f"{mr}_{s_id}"]
                           for s_id in shared_s_ids for mr in p_metric_dict_anysleep}

    # ensure same subjects
    first_mr = list(p_metric_dict_usleep.keys())[0]
    shared_s_ids = set(p_metric_dict_usleep[first_mr]["1"].keys()) & set(gt_metric_dict.keys())
    if set(p_metric_dict_usleep[first_mr]["1"].keys()) != shared_s_ids:
        print(f"Missing keys from pred: {set(p_metric_dict_usleep[first_mr]['1'].keys()) - shared_s_ids}")
    if set(gt_metric_dict.keys()) != shared_s_ids:
        print(f"Missing keys from ground truth: {set(gt_metric_dict.keys()) - shared_s_ids}")
    # filter to shared subjects
    gt_metric_dict = {k: v for k, v in gt_metric_dict.items() if k in shared_s_ids}
    pred_dict_usleep = {
        f"{k}_{k2}": v2
        for k, v in p_metric_dict_usleep.items()
        for k2, v2 in v["1"].items() if k2 in shared_s_ids
    }
    delta_dict_usleep = {f"{mr}_{s_id}": gt_metric_dict[s_id] - pred_dict_usleep[f"{mr}_{s_id}"]
                         for s_id in shared_s_ids for mr in p_metric_dict_usleep}

    if metric_name == "Sleep efficiency":
        table_data.append([
            metric_name,
            get_avg_std(gt_metric_dict, in_hours=False),
            get_avg_std(pred_dict_anysleep, in_hours=False),
            get_avg_std(delta_dict_anysleep, in_hours=False),
            get_spearman(p_metric_dict_anysleep, gt_metric_dict),
            get_avg_std(pred_dict_usleep, in_hours=False),
            get_avg_std(delta_dict_usleep, in_hours=False),
            get_spearman(p_metric_dict_usleep, gt_metric_dict)
        ])
    else:
        table_data.append([
            metric_name,
            get_avg_std(gt_metric_dict, in_hours=True),
            get_avg_std(pred_dict_anysleep, in_hours=True),
            get_avg_std(delta_dict_anysleep, in_hours=True),
            get_spearman(p_metric_dict_anysleep, gt_metric_dict),
            get_avg_std(pred_dict_usleep, in_hours=True),
            get_avg_std(delta_dict_usleep, in_hours=True),
            get_spearman(p_metric_dict_usleep, gt_metric_dict)
        ])

Sleep efficiency
Wake after sleep onset
REM latency
Sleep latency


In [4]:
tabulate(table_data,
         headers=["Metric", "Expert-derived", "AnySleep", "Delta (Exp. - AnySleep)", "Spearman $\\rho$", "U-Sleep",
                  "Delta (Exp. - U-Sleep)",
                  "Spearman $\\rho$"],
         tablefmt="unsafehtml", floatfmt=".2f")

Metric,Expert-derived,AnySleep,Delta (Exp. - AnySleep),Spearman $\rho$,U-Sleep,Delta (Exp. - U-Sleep),Spearman $\rho$
Sleep efficiency,78.80$\pm$14.06,82.38$\pm$13.27,-3.57$\pm$3.57,0.96,81.50$\pm$13.38,-2.70$\pm$3.47,0.96
Wake after sleep onset,1:16$\pm$0:50,1:01$\pm$0:47,0:15$\pm$0:21,0.96,1:04$\pm$0:46,0:12$\pm$0:20,0.94
REM latency,2:06$\pm$1:01,2:05$\pm$1:07,0:00$\pm$0:39,0.87,2:11$\pm$1:09,-0:05$\pm$0:43,0.87
Sleep latency,0:18$\pm$0:20,0:17$\pm$0:24,0:01$\pm$0:19,0.91,0:18$\pm$0:24,-0:00$\pm$0:17,0.89


In [5]:
print(tabulate(table_data,
               headers=["Metric", "Expert-derived", "AnySleep", "Delta (Exp. - AnySleep)", "Spearman $\\rho$",
                        "U-Sleep", "Delta (Exp. - U-Sleep)",
                        "Spearman $\\rho$"],
               tablefmt="latex_raw", floatfmt=".2f"))

\begin{tabular}{llllrllr}
\hline
 Metric                 & Expert-derived   & AnySleep        & Delta (Exp. - AnySleep)   &   Spearman $\rho$ & U-Sleep         & Delta (Exp. - U-Sleep)   &   Spearman $\rho$ \\
\hline
 Sleep efficiency       & 78.80$\pm$14.06  & 82.38$\pm$13.27 & -3.57$\pm$3.57            &              0.96 & 81.50$\pm$13.38 & -2.70$\pm$3.47           &              0.96 \\
 Wake after sleep onset & 1:16$\pm$0:50    & 1:01$\pm$0:47   & 0:15$\pm$0:21             &              0.96 & 1:04$\pm$0:46   & 0:12$\pm$0:20            &              0.94 \\
 REM latency            & 2:06$\pm$1:01    & 2:05$\pm$1:07   & 0:00$\pm$0:39             &              0.87 & 2:11$\pm$1:09   & -0:05$\pm$0:43           &              0.87 \\
 Sleep latency          & 0:18$\pm$0:20    & 0:17$\pm$0:24   & 0:01$\pm$0:19             &              0.91 & 0:18$\pm$0:24   & -0:00$\pm$0:17           &              0.89 \\
\hline
\end{tabular}
